# Feature selection for disability at discharge

In [1]:
# Turn warnings off to keep notebook tidy
import warnings
warnings.filterwarnings("ignore")

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold

## Load data, filter and create k-fold

In [2]:
features_for_selection = [
    'stroke_team',
    'age',
    'male',
    'ethnicity',
    'onset_to_arrival_time',
    'onset_known',
    'precise_onset_known',
    'onset_during_sleep',
    'arrive_by_ambulance',
    'infarction',
    'arrival_to_scan_time',
    'thrombolysis_induced_haemorrhage',
    'thrombolysis',
    'thrombectomy',
    'congestive_heart_failure',
    'hypertension',
    'atrial_fibrillation',
    'diabetes',
    'prior_stroke_tia',
    'afib_antiplatelet',
    'afib_anticoagulant',
    'new_afib_diagnosis',
    'any_afib_diagnosis',
    'prior_disability',
    'stroke_severity',
    'random_1',
    'random_2',
    'random_3'
]

In [3]:
from pathlib import Path

# Resolve data path robustly (works even when Jupyter server uses a different CWD)
candidate_paths = [
    Path("../../data/sam3/cleaned_data.csv"),                 # original relative path
    Path.cwd() / "../../data/sam3/cleaned_data.csv",          # relative to current working dir
    Path.cwd() / "data/sam3/cleaned_data.csv",                # common project-root layout
]

# Also search upward from CWD for a matching data folder
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidate_paths.append(parent / "data/sam3/cleaned_data.csv")

data_path = next((p.resolve() for p in candidate_paths if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        f"Could not find cleaned_data.csv. Current working directory: {Path.cwd()}"
    )

all_data = pd.read_csv(data_path, low_memory=False)

# Add three 'random' columns to the DataFrame with random values for each row
all_data['random_1'] = np.random.rand(len(all_data))
all_data['random_2'] = np.random.binomial(1, 0.5, len(all_data))
all_data['random_3'] = np.random.randint(1, 11, len(all_data))

# List all_data columns
all_data_columns = all_data.columns.tolist()
print(all_data_columns)

# Print size of all_data
print(f"Size of all_data: {all_data.shape}")

['stroke_team', 'age', 'male', 'ethnicity', 'onset_to_arrival_time', 'onset_known', 'precise_onset_known', 'onset_during_sleep', 'arrive_by_ambulance', 'call_to_ambulance_arrival_time', 'ambulance_on_scene_time', 'ambulance_travel_to_hospital_time', 'ambulance_wait_time_at_hospital', 'month', 'year', 'weekday', 'arrival_time_3_hour_period', 'infarction', 'lvo', 'arrival_to_scan_time', 'thrombolysis', 'thrombolysis_induced_haemorrhage', 'scan_to_thrombolysis_time', 'arrival_to_thrombolysis_time', 'onset_to_thrombolysis_time', 'thrombectomy', 'arrival_to_thrombectomy_referral_time', 'onset_to_thrombectomy_time', 'ai_aspects', 'aspects_score', 'perfusion_imaging_used', 'transfer_time', 'arrival_to_thrombectomy_time', 'congestive_heart_failure', 'hypertension', 'atrial_fibrillation', 'diabetes', 'prior_stroke_tia', 'afib_antiplatelet', 'afib_anticoagulant', 'afib_vit_k_anticoagulant', 'afib_doac_anticoagulant', 'afib_heparin_anticoagulant', 'new_afib_diagnosis', 'any_afib_diagnosis', 'prio

Filter data to teams with at least 300 admissions and 10 thrombolysis use

In [4]:
keep = []
groups = all_data.groupby('stroke_team') # creates a new object of groups of data

for index, group_df in groups: # each group has an index and a dataframe of data
    
    # Skip if total admission less than 300 or total thrombolysis < 10
    if (group_df.shape[0] < 300) or (group_df['thrombolysis'].sum() < 10):
        continue
    
    else: 
        keep.append(group_df)

# Concatenate output
data = pd.DataFrame()
data = pd.concat(keep)

n_patients_after_admission_restrictions = data.shape[0]
# Print the number of patients before and after applying the admission restrictions
print(f"Number of patients before admission restrictions: {all_data.shape[0]}")
print(f"Number of patients after admission restrictions: {n_patients_after_admission_restrictions}")
print("Difference in number of patients: ", all_data.shape[0] - n_patients_after_admission_restrictions)

Number of patients before admission restrictions: 499581
Number of patients after admission restrictions: 498754
Difference in number of patients:  827


In [5]:
# Drop any rows with no dicharge_disability value
data = data.dropna(subset=['discharge_disability'])
# Print the number of patients before and after dropping rows with missing discharge_disability
print(f"Number of patients before dropping rows with missing discharge_disability: {n_patients_after_admission_restrictions}")
print(f"Number of patients after dropping rows with missing discharge_disability: {data.shape[0]}")
print("Difference in number of patients: ", n_patients_after_admission_restrictions - data.shape[0])

Number of patients before dropping rows with missing discharge_disability: 498754
Number of patients after dropping rows with missing discharge_disability: 480140
Difference in number of patients:  18614


k-fold splits

In [6]:
# Create 5-fold data stratified by stroke team and discharge disability
strat = data['stroke_team'].map(str) + '-' + data['discharge_disability'].map(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf.get_n_splits(data, strat.values)

# Put in NumPy arrays
X = data.drop(columns=['discharge_disability']).values
y = data['discharge_disability'].values
X_col_names = list(data.drop(columns=['discharge_disability']).columns)
y_col_name = 'discharge_disability'

# Loop through the k-fold splits

train_data = []
test_data = []

counter = 0
for train_index, test_index in skf.split(X, y):  

    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Create a DataFrame for train and test data
    train_df = pd.DataFrame(X_train, columns=X_col_names)
    train_df[y_col_name] = y_train
    
    test_df = pd.DataFrame(X_test, columns=X_col_names)
    test_df[y_col_name] = y_test
    
    # Append to the list
    train_data.append(train_df)
    test_data.append(test_df)

## Single feature prediction

In [ ]:
from pathlib import Path

# Create list to store accuracies and chosen features
roc_auc_by_feature_number = []
ci_lower_by_feature_number = []
ci_upper_by_feature_number = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through available features
for feature in available_features:

    features_to_use = [feature]  # Use only the current feature for this iteration
  
    # Set up a list to hold AUC results for this feature for each kfold
    feature_auc_kfold = []
    
    # Loop through k folds
    for k_fold in range(5):

        # Get k fold split
        train = train_data[k_fold]
        test = test_data[k_fold]

        # Get X and y
        X_train = train.drop('discharge_disability', axis=1)
        X_test = test.drop('discharge_disability', axis=1)
        y_train = train['discharge_disability']
        y_test = test['discharge_disability']

        # Restrict features
        X_train = X_train[features_to_use]
        X_test = X_test[features_to_use]

        # One hot encode hospitals if hospital in features used
        if 'stroke_team' in features_to_use:
            X_train_hosp = pd.get_dummies(
                X_train['stroke_team'], prefix = 'team')
            X_train = pd.concat([X_train, X_train_hosp], axis=1)
            X_train.drop('stroke_team', axis=1, inplace=True)
            X_test_hosp = pd.get_dummies(
                X_test['stroke_team'], prefix = 'team')
            X_test = pd.concat([X_test, X_test_hosp], axis=1)
            X_test.drop('stroke_team', axis=1, inplace=True)

        # One hot encode ethnicity if in features used
        if 'ethnicity' in features_to_use:
            X_train_eth = pd.get_dummies(
                X_train['ethnicity'], prefix = 'eth')
            X_train = pd.concat([X_train, X_train_eth], axis=1)
            X_train.drop('ethnicity', axis=1, inplace=True)
            X_test_eth = pd.get_dummies(
                X_test['ethnicity'], prefix = 'eth')
            X_test = pd.concat([X_test, X_test_eth], axis=1)
            X_test.drop('ethnicity', axis=1, inplace=True)

        # Define model
        model = XGBClassifier(verbosity = 0, seed=42)

        # Fit model
        # Ensure XGBoost-compatible dtypes (the k-fold DataFrames are object-typed)
        X_train = X_train.apply(pd.to_numeric, errors='coerce')
        X_test = X_test.apply(pd.to_numeric, errors='coerce')
        y_train = pd.to_numeric(y_train, errors='coerce')

        model.fit(X_train, y_train)

        # Get predicted y category
        y_pred = model.predict(X_test)

        # Get ROC AUC for predicted categories
        y_proba = model.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
        feature_auc_kfold.append(roc_auc)
    
    # Get average result from all k-fold splits
    feature_auc_mean = np.mean(feature_auc_kfold)
    roc_auc_by_feature_number.append(feature_auc_mean)

    # Get 95% confidence interval for the mean ROC AUC
    ci_lower = np.percentile(feature_auc_kfold, 2.5)
    ci_upper = np.percentile(feature_auc_kfold, 97.5)
    ci_lower_by_feature_number.append(ci_lower)
    ci_upper_by_feature_number.append(ci_upper)

    # Print results for this feature
    print(f"Feature: {feature}, Mean ROC AUC: {feature_auc_mean:.4f}, 95% CI: ({ci_lower:.4f}, {ci_upper:.4f})")

results = pd.DataFrame({
    'Feature': available_features,
    'Mean ROC AUC': roc_auc_by_feature_number,
    'lower_95_ci': ci_lower_by_feature_number,
    'upper_95_ci': ci_upper_by_feature_number
})

# Sort results by Mean ROC AUC in descending order
results = results.sort_values(by='Mean ROC AUC', ascending=False).reset_index(drop=True)

# Save results to CSV
output_dir = Path("./output")
output_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(output_dir / "single_feature_selection_disability_results.csv", index=False)


Feature: stroke_team, Mean ROC AUC: 0.6048, 95% CI: (0.6037, 0.6058)
Feature: age, Mean ROC AUC: 0.6218, 95% CI: (0.6207, 0.6227)
Feature: male, Mean ROC AUC: 0.5391, 95% CI: (0.5389, 0.5394)
Feature: ethnicity, Mean ROC AUC: 0.5075, 95% CI: (0.5070, 0.5081)
Feature: onset_to_arrival_time, Mean ROC AUC: 0.5275, 95% CI: (0.5268, 0.5283)
Feature: onset_known, Mean ROC AUC: 0.5045, 95% CI: (0.5034, 0.5054)
Feature: precise_onset_known, Mean ROC AUC: 0.5174, 95% CI: (0.5168, 0.5179)
Feature: onset_during_sleep, Mean ROC AUC: 0.5074, 95% CI: (0.5069, 0.5078)
Feature: arrive_by_ambulance, Mean ROC AUC: 0.5718, 95% CI: (0.5710, 0.5723)
Feature: infarction, Mean ROC AUC: 0.5327, 95% CI: (0.5321, 0.5331)


In [ ]:
results

## Feature selection

In [ ]:
# Create list to store accuracies and chosen features
roc_auc_by_feature_number = []
roc_auc_by_feature_number_kfold = []
chosen_features = []

num_features = len(features_for_selection)
available_features = features_for_selection.copy()

# Loop through number of features
for i in range (num_features):
    
    # Reset best feature and accuracy
    best_result = 0
    best_feature = ''
    
    # Loop through available features
    for feature in available_features:

        # Create copy of already chosen features to avoid original being changed
        features_to_use = chosen_features.copy()
        # Create a list of features from features already chosen + 1 new feature
        features_to_use.append(feature)
        
        # Set up a list to hold AUC results for this feature for each kfold
        feature_auc_kfold = []
        
        # Loop through k folds
        for k_fold in range(5):

            # Get k fold split
            train = train_data[k_fold]
            test = test_data[k_fold]

            # Get X and y
            X_train = train.drop('discharge_disability', axis=1)
            X_test = test.drop('discharge_disability', axis=1)
            y_train = train['discharge_disability']
            y_test = test['discharge_disability']

            # Restrict features
            X_train = X_train[features_to_use]
            X_test = X_test[features_to_use]
    
            # One hot encode hospitals if hospital in features used
            if 'stroke_team' in features_to_use:
                X_train_hosp = pd.get_dummies(
                    X_train['stroke_team'], prefix = 'team')
                X_train = pd.concat([X_train, X_train_hosp], axis=1)
                X_train.drop('stroke_team', axis=1, inplace=True)
                X_test_hosp = pd.get_dummies(
                    X_test['stroke_team'], prefix = 'team')
                X_test = pd.concat([X_test, X_test_hosp], axis=1)
                X_test.drop('stroke_team', axis=1, inplace=True)

            # One hot encode ethnicity if in features used
            if 'ethnicity' in features_to_use:
                X_train_eth = pd.get_dummies(
                    X_train['ethnicity'], prefix = 'eth')
                X_train = pd.concat([X_train, X_train_eth], axis=1)
                X_train.drop('ethnicity', axis=1, inplace=True)
                X_test_eth = pd.get_dummies(
                    X_test['ethnicity'], prefix = 'eth')
                X_test = pd.concat([X_test, X_test_eth], axis=1)
                X_test.drop('ethnicity', axis=1, inplace=True)

            # Define model
            model = XGBClassifier(verbosity = 0, seed=42)

            # Fit model
            # Ensure XGBoost-compatible dtypes (the k-fold DataFrames are object-typed)
            X_train = X_train.apply(pd.to_numeric, errors='coerce')
            X_test = X_test.apply(pd.to_numeric, errors='coerce')
            y_train = pd.to_numeric(y_train, errors='coerce')

            model.fit(X_train, y_train)

            # Get predicted y category
            y_pred = model.predict(X_test)

            # Get ROC AUC for predicted categories
            y_proba = model.predict_proba(X_test)
            roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            feature_auc_kfold.append(roc_auc)
        
        # Get average result from all k-fold splits
        feature_auc_mean = np.mean(feature_auc_kfold)
    
        # Update chosen feature and result if this feature is a new best
        if feature_auc_mean > best_result:
            best_result = feature_auc_mean
            best_result_kfold = feature_auc_kfold
            best_feature = feature
            
    # k-fold splits are complete    
    # Add mean accuracy and AUC to record of accuracy by feature number
    roc_auc_by_feature_number.append(best_result)
    roc_auc_by_feature_number_kfold.append(best_result_kfold)
    chosen_features.append(best_feature)
    available_features.remove(best_feature)
            
    print (f'Feature {i+1:2.0f}: {best_feature}, AUC: {best_result:0.3f}')

In [ ]:
# Create a DataFrame to hold the results
results_df = pd.DataFrame({
    'Feature Number': list(range(1, num_features + 1)),
    'Chosen Feature': chosen_features,
    'Mean AUC': roc_auc_by_feature_number,
})

# Save to output folder
output_dir = Path.cwd() / "output"
output_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_dir / "feature_selection_disability_results.csv", index=False)

In [ ]:
results_df